# 循环神经网络（上）SRN / LSTM 与梯度截断

本节用一个**数字记忆任务**比较 SRN（最简单循环网络）和 LSTM 的长程依赖能力，并演示梯度截断。

**任务**：输入长度 $L$ 的数字序列（每位 ∈ {0..9}），预测**前两位之和**（0..18 共 19 类）。模型必须把开头信息一直保留到序列末尾。

## 1. 数据

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

def make_digit_sum(n, L, seed=0):
    g = torch.Generator().manual_seed(seed)
    X = torch.randint(0, 10, (n, L), generator=g)
    y = (X[:, 0] + X[:, 1]).long()
    return X, y

X, y = make_digit_sum(5, L=10)
for xi, yi in zip(X, y):
    print(f'seq {xi.tolist()}  ->  label {yi.item()} (= {xi[0].item()}+{xi[1].item()})')

## 2. SRN 从头实现

递推：$h_t = \tanh(W_x x_t + W_h h_{t-1} + b)$。

In [ ]:
class MySRN(nn.Module):
    def __init__(self, vocab=10, embed=32, hidden=64, n_class=19):
        super().__init__()
        self.emb = nn.Embedding(vocab, embed)
        self.Wx = nn.Parameter(torch.randn(embed,  hidden) / math.sqrt(embed))
        self.Wh = nn.Parameter(torch.eye(hidden))                    # identity init：给 SRN 加 buff，保护长程信息；
        # 实战中更常见的是 randn / sqrt(hidden)，效果会显著差于 LSTM
        self.b  = nn.Parameter(torch.zeros(hidden))
        self.fc = nn.Linear(hidden, n_class)

    def forward(self, x):                  # x: [B, L]
        e = self.emb(x)                    # [B, L, embed]
        B, L, _ = e.shape
        h = torch.zeros(B, self.Wh.size(0))
        for t in range(L):
            h = torch.tanh(e[:, t] @ self.Wx + h @ self.Wh + self.b)
        return self.fc(h)

print('output shape:', MySRN()(torch.zeros(2, 10, dtype=torch.long)).shape)

### 与 `nn.RNN` 对齐

PyTorch 的 `nn.RNN` 权重布局：`weight_ih_l0` 形状 `[hidden, input]`、`weight_hh_l0` 形状 `[hidden, hidden]`——与手写的 `Wx (input, hidden)`、`Wh (hidden, hidden)` 互为转置。

In [ ]:
torch.manual_seed(0)
embed_dim, hidden_dim = 16, 32
emb = nn.Embedding(10, embed_dim)

Wx = torch.randn(embed_dim, hidden_dim) / math.sqrt(embed_dim)
Wh = torch.eye(hidden_dim)
b  = torch.zeros(hidden_dim)

x = torch.randint(0, 10, (2, 5))
e = emb(x)
h = torch.zeros(2, hidden_dim)
for t in range(5):
    h = torch.tanh(e[:, t] @ Wx + h @ Wh + b)
h_manual = h

rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True, nonlinearity='tanh')
with torch.no_grad():
    rnn.weight_ih_l0.copy_(Wx.T)
    rnn.weight_hh_l0.copy_(Wh.T)
    # 注意：nn.RNN 的有效 bias = bias_ih + bias_hh；这里手写 b=zeros，所以两边都置零
    # 如果未来把手写 b 改成非零值，需要相应分配（例如 bias_ih = b, bias_hh = zeros）
    rnn.bias_ih_l0.zero_()
    rnn.bias_hh_l0.zero_()
_, h_final = rnn(e)
print('max abs diff:', (h_manual - h_final.squeeze(0)).abs().max().item())

## 3. LSTM

用 `nn.LSTM` 直接调即可。接口与 `nn.RNN` 类似，返回 `(out, (h, c))`。

In [ ]:
class MyLSTMModel(nn.Module):
    def __init__(self, vocab=10, embed=32, hidden=64, n_class=19):
        super().__init__()
        self.emb = nn.Embedding(vocab, embed)
        self.lstm = nn.LSTM(embed, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, n_class)

    def forward(self, x):
        e = self.emb(x)
        _, (h, _) = self.lstm(e)
        return self.fc(h.squeeze(0))

## 4. 训练对比：SRN vs LSTM × 不同序列长度

在 $L \in \{10, 20\}$ 上各训 20 个 epoch，统一用 Adam + grad clip。

In [ ]:
def train(model_factory, L, epochs=20, lr=3e-3, n_train=3000, clip=1.0):
    torch.manual_seed(0)
    model = model_factory()
    opt = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    X_tr, y_tr = make_digit_sum(n_train, L, seed=0)
    X_dv, y_dv = make_digit_sum(500,     L, seed=1)
    train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=128, shuffle=True)
    accs = []
    for _ in range(epochs):
        model.train()
        for x, y in train_loader:
            opt.zero_grad()
            loss_fn(model(x), y).backward()
            if clip is not None:
                nn.utils.clip_grad_norm_(model.parameters(), clip)
            opt.step()
        model.eval()
        with torch.no_grad():
            acc = (model(X_dv).argmax(1) == y_dv).float().mean().item()
        accs.append(acc)
    return accs

results = {}
for L in [10, 20]:
    print(f'--- L = {L} ---')
    results[('SRN',  L)] = train(MySRN,        L)
    print(f'  SRN  best={max(results[("SRN",  L)]):.3f}  final={results[("SRN",  L)][-1]:.3f}')
    results[('LSTM', L)] = train(MyLSTMModel,  L)
    print(f'  LSTM best={max(results[("LSTM", L)]):.3f}  final={results[("LSTM", L)][-1]:.3f}')

### 训练曲线

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, L in zip(axes, [10, 20]):
    ax.plot(results[('SRN',  L)], label='SRN')
    ax.plot(results[('LSTM', L)], label='LSTM')
    ax.set_title(f'seq_len = {L}')
    ax.set_xlabel('epoch'); ax.set_ylabel('dev accuracy'); ax.set_ylim(0, 1.05)
    ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

**观察**：

- L=10 时 LSTM 学到接近 100%，SRN 只能到 ~70%——LSTM 的门机制让长程信息更容易保留。
- L=20 时**两者都几乎不学**（接近 1/19 ≈ 5% 的随机水平）。LSTM 缓解了长程依赖问题，但没有消除——这正是 chap8 注意力机制要解决的痛点。

## 5. 梯度截断：让训练更稳

RNN 的 BPTT 会经过 $L$ 步反向传播，梯度容易在某些 step 突然变大。`nn.utils.clip_grad_norm_(params, max_norm)` 会在反传完成后、`opt.step()` 之前，把所有参数梯度的整体 L2 范数缩放到 ≤ `max_norm`。

下面用一个随机初始化（**非 identity**）的 SRN 跑长序列，记录每个 step 的梯度范数。

In [ ]:
class WildSRN(nn.Module):
    """Wh 用更大的随机初始化（不是 identity），更容易看到梯度变大的情形。"""
    def __init__(self, embed=32, hidden=64, n_class=19):
        super().__init__()
        self.emb = nn.Embedding(10, embed)
        self.Wx = nn.Parameter(torch.randn(embed,  hidden) / math.sqrt(embed))
        self.Wh = nn.Parameter(torch.randn(hidden, hidden) * 2.0 / math.sqrt(hidden))
        self.b  = nn.Parameter(torch.zeros(hidden))
        self.fc = nn.Linear(hidden, n_class)

    def forward(self, x):
        e = self.emb(x); h = torch.zeros(e.size(0), self.Wh.size(0))
        for t in range(e.size(1)):
            h = torch.tanh(e[:, t] @ self.Wx + h @ self.Wh + self.b)
        return self.fc(h)

@torch.no_grad()
def grad_l2(model):
    s = 0.0
    for p in model.parameters():
        if p.grad is not None:
            s += p.grad.pow(2).sum().item()
    return math.sqrt(s)

def grad_trace(clip, epochs=2, L=50):
    torch.manual_seed(0)
    model = WildSRN()
    opt = optim.SGD(model.parameters(), lr=1.0)
    loss_fn = nn.CrossEntropyLoss()
    X_tr, y_tr = make_digit_sum(1500, L, seed=0)
    loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=64, shuffle=True)
    trace = []
    for _ in range(epochs):
        for x, y in loader:
            opt.zero_grad(); loss_fn(model(x), y).backward()
            trace.append(grad_l2(model))
            if clip is not None:
                nn.utils.clip_grad_norm_(model.parameters(), clip)
            opt.step()
    return trace

g_no   = grad_trace(clip=None)
g_clip = grad_trace(clip=0.5)

plt.figure(figsize=(9, 4))
plt.plot(g_no,   label='no clip')
plt.plot(g_clip, label='clip=0.5')
plt.axhline(0.5, color='red', lw=0.5, ls='--')
plt.xlabel('step'); plt.ylabel('grad L2 norm before clipping'); plt.yscale('log')
plt.legend(); plt.grid(alpha=0.3); plt.show()
print(f'no clip : max={max(g_no):.3e}   median={sorted(g_no)[len(g_no)//2]:.3e}')
print(f'clip 0.5: max={max(g_clip):.3e}   median={sorted(g_clip)[len(g_clip)//2]:.3e}')
print('(注意：trace 记录的是 clip 之前的值；clip 影响的是 *实际写入参数* 的更新量。)')

**说明**：横坐标是 step，纵坐标是**截断前**的梯度 L2 范数（log 轴）。`clip=0.5` 把所有更新步的范数实际写入到优化器之前压到 0.5 以下。`no clip` 偶尔会冒出尖峰甚至更高的值——这就是为什么 RNN/Transformer 训练都默认带 grad clip。

## 小结

- **手写 SRN 与 `nn.RNN` 等价**——但实际项目用 `nn.RNN / nn.LSTM / nn.GRU`，底层 fused 实现快得多。
- **`weight_ih_l0` / `weight_hh_l0`** 的形状是 `[hidden, input/hidden]`，与教科书 / 手写公式互为转置。
- **LSTM > SRN**，但**两者在 L=20+ 都吃力**——长程依赖是 RNN 家族的根本难题。
- **`clip_grad_norm_(params, max_norm)`** 是 RNN/Transformer 训练的标配，几乎不用考虑是否打开。
- chap6 下：用双向 LSTM 做序列分类，配合 `nn.utils.rnn.pack_padded_sequence` 处理变长序列。